Human‑in‑the‑Loop
Meaning: A human can step in during the agent’s reasoning process to guide, approve, or correct its actions.
Example: An agent is writing SQL queries. Before it runs a risky query, it asks you to confirm — you’re the “human in the loop.”

User
 ↓
LLM decides tool is needed
 ↓
Pause for human approval
 ↓
Approved?
 ↓
Yes → Execute tool
No  → Stop

Checkpointing:
Meaning: Saving the agent’s state at certain steps so you can roll back or resume later.
Example: An agent is processing a long workflow. If it crashes halfway, checkpointing lets it restart from the last saved step instead of starting over.



chatbot completed
 ↓
State saved

approval completed
 ↓
State saved

delete_tool completed
 ↓
State saved

If the application crashes after the approval step:

Restart app
 ↓
Load thread_id = session_1
 ↓
Resume from last checkpoint
 ↓
Continue execution

Parallel Fan‑Out:
Meaning: Running multiple tasks at the same time instead of one after another.
Example: An agent needs to fetch data from 3 APIs. Instead of calling them one by one (slow), it “fans out” and queries all 3 at once, then merges the answers.

In [45]:
import sqlite3
from typing import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.checkpoint.memory import MemorySaver

class GraphState(TypedDict):
    input: str
    approved: bool
    result: str

def approval_node(state: GraphState) -> GraphState:
    query = state["input"]
    print(f"⚠️ Approval needed: Do you want to run this query? {query}")
    user_input = input("Type 'yes' to approve, anything else to reject: ")
    return {**state, "approved": user_input.lower() == "yes"}

def db_query_node(state: GraphState) -> GraphState:
    if not state["approved"]:
        return {**state, "result": "Query rejected"}
    try:
        conn = sqlite3.connect("example.db")
        cursor = conn.cursor()
        cursor.execute(state["input"])
        results = cursor.fetchall()
        conn.close()
        return {**state, "result": str(results)}
    except Exception as e:
        return {**state, "result": f"Error: {e}"}

checkpointer = MemorySaver()
workflow = StateGraph(GraphState)
workflow.add_node("approval", approval_node)
workflow.add_node("db_query", db_query_node)
workflow.add_edge(START, "approval")
workflow.add_edge("approval", "db_query")
app = workflow.compile(checkpointer=checkpointer)

# Prepare DB
conn = sqlite3.connect("example.db")
cursor = conn.cursor()
cursor.execute("CREATE TABLE IF NOT EXISTS users (id INTEGER, name TEXT)")
cursor.execute("DELETE FROM users")
cursor.execute("INSERT INTO users VALUES (1, 'Alice')")
cursor.execute("INSERT INTO users VALUES (2, 'Bob')")
conn.commit()
conn.close()

# Run workflow
state = app.invoke(
    {"input": "SELECT COUNT(*) FROM users", "approved": False, "result": ""},
    config={"configurable": {"thread_id": "lab1"}}
)
print("First run:", state)


⚠️ Approval needed: Do you want to run this query? SELECT COUNT(*) FROM users
First run: {'input': 'SELECT COUNT(*) FROM users', 'approved': True, 'result': '[(2,)]'}


In [46]:
saved_state = app.get_state(config={"configurable": {"thread_id": "lab1"}})
print("Saved checkpoint:", saved_state)


Saved checkpoint: StateSnapshot(values={'input': 'SELECT COUNT(*) FROM users', 'approved': True, 'result': '[(2,)]'}, next=(), config={'configurable': {'thread_id': 'lab1', 'checkpoint_ns': '', 'checkpoint_id': '1f16e417-bc6a-6f51-8002-f35b0938d2de'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-06-22T13:51:35.374395+00:00', parent_config={'configurable': {'thread_id': 'lab1', 'checkpoint_ns': '', 'checkpoint_id': '1f16e417-bc1b-60fb-8001-0359c6d912cb'}}, tasks=(), interrupts=())


In [47]:
state2 = app.invoke(
    {"input": "SELECT * FROM users", "approved": True, "result": ""},
    config={"configurable": {"thread_id": "lab1"}}
)
print("Second run (resumed):", state2)


⚠️ Approval needed: Do you want to run this query? SELECT * FROM users
Second run (resumed): {'input': 'SELECT * FROM users', 'approved': True, 'result': "[(1, 'Alice'), (2, 'Bob')]"}


Because you’re using the same thread_id, the checkpointer ties both runs together. If you switch to a new thread_id, it starts fresh.